# 05 — Brushstroke ordering and coloring (Porter-Duff)

Paper Algorithm 2 + Eq. 8. For a pair of overlapping brushstrokes `A` and `B` detected in notebook 04, we don't know which one was painted first. This notebook infers the ordering by trying both compositing directions, solving for the stroke colors under each assumption, and picking the assumption that reconstructs the observed pixels with lower residual.

Inputs: `outputs/dstroke/soft/{artist}/{stem}.npz` from `04_soft_segmentation.ipynb`.

In [ ]:
from __future__ import annotations
import sys
from pathlib import Path

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

NB_DIR = Path.cwd()
sys.path.insert(0, str(NB_DIR))
import dstroke_utils as d

REPO_ROOT = Path('C:/Users/MichelleJacobs/Repos/AIAi')
RAW = REPO_ROOT / 'AIA - AI Tool - Fingerprinting-1' / 'data' / 'raw'
OUT = REPO_ROOT / 'AIA - AI Tool - Fingerprinting-1' / 'outputs' / 'dstroke'
SOFT = OUT / 'soft'

## Synthetic validation

Before running on real paintings, verify Algorithm 2 recovers the correct order on a controlled synthetic scenario where we know the ground truth.

In [ ]:
def gauss_blob(h, w, cy, cx, sy, sx, peak=0.9):
    ys, xs = np.mgrid[:h, :w]
    z = np.exp(-(((ys - cy)**2 / (2*sy*sy)) + ((xs - cx)**2 / (2*sx*sx))))
    return (z * peak).astype(np.float32)

H, W = 128, 128
aA = gauss_blob(H, W, 55, 55, 18, 18)
aB = gauss_blob(H, W, 70, 70, 18, 18)
colA = np.array([0.85, 0.15, 0.20])   # warm red
colB = np.array([0.10, 0.35, 0.75])   # cool blue

denom = np.clip(aA + (1-aA)*aB, 1e-6, None)
obs = (aA[..., None]*colA + ((1-aA)*aB)[..., None]*colB) / denom[..., None]
obs = np.clip(obs, 0, 1)

maskA = aA > 0.1; maskB = aB > 0.1
result = d.porter_duff_ordering(obs, maskA, maskB, aA, aB)
print('ground truth: A over B | recovered:', result['order'])
print(f'residual correct:  {result["residual_correct"]:.5f}')
print(f'residual reversed: {result["residual_reversed"]:.5f}')
print('true A rgb:', colA, ' recovered:', result['color_A'])
print('true B rgb:', colB, ' recovered:', result['color_B'])

fig, ax = plt.subplots(1, 3, figsize=(9, 3))
ax[0].imshow(obs); ax[0].set_title('observed A over B')
ax[1].imshow(np.stack([aA]*3, axis=-1)); ax[1].set_title('alpha A')
ax[2].imshow(np.stack([aB]*3, axis=-1)); ax[2].set_title('alpha B')
for a in ax: a.set_xticks([]); a.set_yticks([])
plt.show()

## Apply to real paintings

Find the overlapping stroke pairs in each saved `.npz`: two strokes `i, j` overlap if `min(α_i, α_j) > 0.3` on ≥ 30 pixels. Sort by overlap area and process the top pairs.

In [ ]:
def find_overlapping_pairs(alphas, min_overlap=30, max_pairs=5):
    pairs = []
    n = alphas.shape[0]
    for i in range(n):
        for j in range(i+1, n):
            overlap = np.minimum(alphas[i], alphas[j]) > 0.3
            c = int(overlap.sum())
            if c >= min_overlap:
                pairs.append((i, j, c))
    pairs.sort(key=lambda t: -t[2])
    return pairs[:max_pairs]

def process_painting(npz_path: Path, artist: str):
    alphas = np.load(npz_path)['alphas']
    # find the matching source painting for its RGB
    cand = list((RAW / artist / 'authenticated').glob(f'{npz_path.stem}.*'))
    if not cand:
        print(f'no source painting for {npz_path.stem}'); return
    rgb = np.array(Image.open(cand[0]).convert('RGB').resize((256, 256)), dtype=np.float32) / 255.0

    pairs = find_overlapping_pairs(alphas)
    if not pairs:
        print(f'{npz_path.stem}: no overlapping pairs found'); return

    fig, axes = plt.subplots(len(pairs), 4, figsize=(10, 2.2 * len(pairs)))
    if len(pairs) == 1: axes = axes[None, :]

    for r, (i, j, c) in enumerate(pairs):
        aA = alphas[i]; aB = alphas[j]
        maskA = aA > 0.3; maskB = aB > 0.3
        try:
            result = d.porter_duff_ordering(rgb, maskA, maskB, aA, aB)
        except ValueError:
            continue
        axes[r, 0].imshow(rgb); axes[r, 0].set_title('painting')
        comb = np.zeros_like(rgb); comb[..., 0] = aA; comb[..., 2] = aB
        axes[r, 1].imshow(comb); axes[r, 1].set_title(f'A (red) / B (blue), overlap={c}')
        axes[r, 2].imshow(np.ones_like(rgb) * result['color_A']); axes[r, 2].set_title(f'A rgb (order={result["order"]})')
        axes[r, 3].imshow(np.ones_like(rgb) * result['color_B']); axes[r, 3].set_title(f'B rgb (resid {result["residual_correct"]:.4f})')
        for a in axes[r]: a.set_xticks([]); a.set_yticks([])
    fig.suptitle(f'{artist}: {npz_path.name}')
    fig.tight_layout(); plt.show()

# process one painting per artist if present
for artist in ('manet', 'van_gogh', 'rembrandt'):
    art_dir = SOFT / artist
    if not art_dir.exists(): continue
    for npz in sorted(art_dir.glob('*.npz'))[:1]:
        process_painting(npz, artist)

## Notes

- Paper Remark (§3.3): recovered per-stroke color + ordering are intended to seed layer-decomposition optimization in other methods. For authentication, the distribution of *residuals* across many pairs may itself be a useful feature — e.g., a painter who works alla-prima will have a different residual profile than one who builds up many glazes.
- If ordering looks unstable, increase `unknown_band` in `04_soft_segmentation.ipynb` — a wider unknown zone gives the guided filter more room to recover accurate soft transitions.
- If the recovered RGB falls outside `[0, 1]` and gets clipped, the paper's least-squares assumption (solid-color stroke with only transparency variation) is likely violated in that pair — flag those pairs as low-confidence.